# 🛡️ Evaluación por Zonas — Auditor RoBERTa Clínico

Evalúa el **mejor modelo** (RoBERTa clínico + cross-entropy ponderada, F1-macro 7 clases = 0,5871) a nivel de **zona clínica**, que es el objetivo real del sistema:

- **Zona segura:** BI-RADS 0, 1, 2, 3 (control/seguimiento)
- **Zona de riesgo:** BI-RADS 4, 5, 6 (biopsia/manejo oncológico)

No se reentrena nada: se carga el modelo guardado (`mejor_auditor_roberta.pt`) y se reevalúa el **mismo conjunto de prueba real** (semilla 42). La métrica clave es la **sensibilidad de la zona de riesgo** (de los informes que de verdad eran riesgo, cuántos detectó).

## 1 · Librerías y configuración

In [1]:
import torch, torch.nn as nn, numpy as np, pandas as pd, sys
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score
sys.path.append('..')
from src.preprocessing import MAX_LENGTH

SEED=42
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO     = 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'
MODELO_ALT = 'BSC-LT/roberta-base-biomedical-clinical-es'
RUTA = '../results/mejor_auditor_roberta.pt'   # mejor modelo (cross-entropy)
print(f"Dispositivo: {DEVICE}")

Dispositivo: mps


## 2 · Cargar datos y reconstruir el conjunto de prueba

Mismo split estratificado (semilla 42) para recuperar exactamente el test real usado en el entrenamiento.

In [2]:
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X = df['auditor_input'].values
y = df['BI-RADS'].values
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
print(f"Test: {len(X_test)} informes")
print("Distribución test por BI-RADS:")
print(pd.Series(y_test).value_counts().sort_index().to_string())

Test: 654 informes
Distribución test por BI-RADS:
0    145
1     89
2    396
3     13
4      8
5      2
6      1


## 3 · Tokenizador, Dataset y modelo

In [3]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODELO)
except Exception:
    MODELO = MODELO_ALT; tokenizer = AutoTokenizer.from_pretrained(MODELO)

class BIRADSDataset(Dataset):
    def __init__(self, textos, labels, tok, max_length=MAX_LENGTH):
        self.t=list(textos); self.l=list(labels); self.tok=tok; self.ml=max_length
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),truncation=True,padding='max_length',max_length=self.ml,
                   return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}

class AuditorRoBERTa(nn.Module):
    def __init__(self, modelo, n_classes=7, dropout=0.5):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(modelo)
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(self.encoder.config.hidden_size, n_classes)
    def forward(self, input_ids, attention_mask):
        out=self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(out.last_hidden_state[:,0,:]))

test_loader = DataLoader(BIRADSDataset(X_test, y_test, tokenizer), batch_size=16)
model = AuditorRoBERTa(MODELO).to(DEVICE)
model.load_state_dict(torch.load(RUTA, map_location=DEVICE))
model.eval()
print("Modelo cargado desde", RUTA)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado desde ../results/mejor_auditor_roberta.pt


## 4 · Predicción en test (7 clases)

In [4]:
preds7, reales7 = [], []
with torch.no_grad():
    for batch in test_loader:
        ids=batch['input_ids'].to(DEVICE); mask=batch['attention_mask'].to(DEVICE)
        preds7.extend(model(ids,mask).argmax(1).cpu().numpy())
        reales7.extend(batch['labels'].numpy())
preds7=np.array(preds7); reales7=np.array(reales7)
print("Predicciones generadas:", len(preds7))

Predicciones generadas: 654


## 5 · Mapeo a zonas y métricas clínicas

Zona segura (0–3) → 0 · Zona de riesgo (4–6) → 1. La clase positiva es **riesgo**.

In [5]:
def a_zona(v):  # 0-3 -> 0 (segura), 4-6 -> 1 (riesgo)
    return (np.asarray(v) >= 4).astype(int)

z_real = a_zona(reales7)
z_pred = a_zona(preds7)

cm = confusion_matrix(z_real, z_pred, labels=[0,1])
tn, fp, fn, tp = cm.ravel()

sensibilidad = tp/(tp+fn) if (tp+fn) else 0.0   # recall riesgo (detectar lo grave)
especificidad= tn/(tn+fp) if (tn+fp) else 0.0
vpp          = tp/(tp+fp) if (tp+fp) else 0.0    # precisión en riesgo
vpn          = tn/(tn+fn) if (tn+fn) else 0.0
f1_riesgo    = f1_score(z_real, z_pred, pos_label=1, zero_division=0)
f1_macro_z   = f1_score(z_real, z_pred, average='macro', zero_division=0)
acc          = (z_real==z_pred).mean()

print("="*55)
print("🛡️  EVALUACIÓN POR ZONAS — TEST SET")
print("="*55)
print(f"  Informes en zona de riesgo (reales): {int((z_real==1).sum())}")
print(f"  Informes en zona segura  (reales):   {int((z_real==0).sum())}")
print("-"*55)
print(f"  Sensibilidad (recall riesgo): {sensibilidad:.4f}  <- clave clínica")
print(f"  Especificidad:                {especificidad:.4f}")
print(f"  VPP (precisión riesgo):       {vpp:.4f}")
print(f"  VPN:                          {vpn:.4f}")
print(f"  F1 zona de riesgo:            {f1_riesgo:.4f}")
print(f"  F1-macro (2 zonas):           {f1_macro_z:.4f}")
print(f"  Accuracy:                     {acc:.4f}")
print("="*55)

🛡️  EVALUACIÓN POR ZONAS — TEST SET
  Informes en zona de riesgo (reales): 11
  Informes en zona segura  (reales):   643
-------------------------------------------------------
  Sensibilidad (recall riesgo): 0.5455  <- clave clínica
  Especificidad:                0.9938
  VPP (precisión riesgo):       0.6000
  VPN:                          0.9922
  F1 zona de riesgo:            0.5714
  F1-macro (2 zonas):           0.7822
  Accuracy:                     0.9862


## 6 · Matriz de confusión por zonas

In [6]:
print("                 PREDICHO")
print("                 Segura  Riesgo")
print(f"REAL  Segura   |   {tn:4d}    {fp:4d}")
print(f"      Riesgo   |   {fn:4d}    {tp:4d}")
print()
print(f"Interpretación clínica:")
print(f"  • {tp} informes de riesgo correctamente detectados (verdaderos positivos)")
print(f"  • {fn} informes de riesgo NO detectados (falsos negativos) <- los que preocupan")
print(f"  • {fp} falsas alarmas (zona segura marcada como riesgo)")
print(f"  • {tn} informes seguros correctamente identificados")

                 PREDICHO
                 Segura  Riesgo
REAL  Segura   |    639       4
      Riesgo   |      5       6

Interpretación clínica:
  • 6 informes de riesgo correctamente detectados (verdaderos positivos)
  • 5 informes de riesgo NO detectados (falsos negativos) <- los que preocupan
  • 4 falsas alarmas (zona segura marcada como riesgo)
  • 639 informes seguros correctamente identificados


## 7 · Cómo reportar esto

- **Reportar ambas métricas:** el F1-macro a 7 categorías (0,5871) por transparencia técnica, y el desempeño por zonas por relevancia clínica. No esconder la dificultad fina.
- La **sensibilidad de zona de riesgo** responde a la pregunta clínica central: ¿cuántos casos potencialmente graves detecta el sistema? Es la métrica que más le importa a un radiólogo.
- Recordar la limitación honesta: la zona de riesgo tiene pocos casos reales en test, por lo que estas métricas, aunque más estables que a 7 clases, siguen teniendo intervalos de confianza amplios. Es una **prueba de concepto**, no una validación definitiva.